# Marker PDF Converter + NVIDIA API (Google Colab)

Converts PDF, images, DOCX, PPTX, XLSX, HTML and EPUB to Markdown/HTML/JSON with [Marker](https://github.com/VikParuchuri/marker), optionally using NVIDIA's free API as the LLM. The result is downloaded as a ZIP ready to unzip into your [Obsidian](https://obsidian.md) vault.

**Run the cells in order, top to bottom.** In step 5 you choose whether to convert **one document or several**: just select one or more files in the upload dialog. If the runtime restarts, start again from step 2.

Before you start: *Runtime → Change runtime type → T4 GPU*. If you will use the LLM, get your API key at [build.nvidia.com](https://build.nvidia.com) (it starts with `nvapi-`).

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install Marker (pinned version)

Installs `marker-pdf 1.10.2`. **Do not use 2.0**: its layout models need Docker (vLLM) or llama.cpp, which are not available in Colab, and it fails with `docker binary not found`. This takes a few minutes.

In [ ]:
!pip install -q "marker-pdf[full]==1.10.2" "surya-ocr>=0.17.1,<0.18" openai

from importlib.metadata import version
mv, sv = version("marker-pdf"), version("surya-ocr")
print("marker-pdf:", mv, "| surya-ocr:", sv)
assert mv.startswith("1."), "The wrong version of marker-pdf was installed"
print("✓ Marker installed")

## 3. Options

| Option | What it does |
|---|---|
| `USE_LLM` | Uses the NVIDIA model to improve tables, equations and text. Slower, and needs an API key. |
| `NVIDIA_MODEL` | Multimodal (text + image) model from build.nvidia.com. |
| `OUTPUT_FORMAT` | `markdown` builds the Obsidian package; `html` and `json` are downloaded exactly as Marker creates them. |
| `FORCE_OCR` | Turn on only if the text comes out garbled or the PDF is scanned. |
| `PAGE_RANGE` | Pages to convert, starting at 0. E.g. `0-10` or `0,5-10`. Empty = all. Applies to every document. |
| `REDO_INLINE_MATH` | Redoes the equations inside the text (only with `USE_LLM`). |

In [ ]:
USE_LLM          = False                  # @param {type:"boolean"}
NVIDIA_MODEL     = "z-ai/glm-5.3-flash"   # @param {type:"string"}
OUTPUT_FORMAT    = "markdown"             # @param ["markdown", "html", "json"]
FORCE_OCR        = False                  # @param {type:"boolean"}
PAGE_RANGE       = ""                     # @param {type:"string"}
REDO_INLINE_MATH = True                   # @param {type:"boolean"}

NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
PAGE_RANGE = PAGE_RANGE.strip()

print(f"LLM: {'yes (' + NVIDIA_MODEL + ')' if USE_LLM else 'no'} | format: {OUTPUT_FORMAT} | "
      f"forced OCR: {'yes' if FORCE_OCR else 'no'} | pages: {PAGE_RANGE or 'all'}")

## 4. Connect to the NVIDIA API (only if `USE_LLM = True`)

Reads the key from the Colab secret `NVIDIA_API_KEY` (key icon in the sidebar) or asks you for it. Then it tests the model with text and with an image. Without the LLM, this cell does nothing.

In [ ]:
from getpass import getpass

NVIDIA_API_KEY = ""

if not USE_LLM:
    print("No-LLM mode: no API key needed. Go to step 5.")
else:
    try:
        from google.colab import userdata
        NVIDIA_API_KEY = (userdata.get("NVIDIA_API_KEY") or "").strip()
        print("✓ Key read from the Colab secret NVIDIA_API_KEY")
    except Exception:
        pass
    if not NVIDIA_API_KEY:
        NVIDIA_API_KEY = getpass("Paste your NVIDIA API key (nvapi-...): ").strip()

    import base64, io
    from openai import OpenAI
    from PIL import Image, ImageDraw

    client = OpenAI(base_url=NVIDIA_BASE_URL, api_key=NVIDIA_API_KEY)
    try:
        r = client.chat.completions.create(
            model=NVIDIA_MODEL,
            messages=[{"role": "user", "content": "Reply only: OK"}],
            max_tokens=300,
        )
        print("✓ Text OK. Reply:", (r.choices[0].message.content or "").strip()[:80])

        img = Image.new("RGB", (300, 100), "white")
        ImageDraw.Draw(img).text((20, 40), "HELLO 42", fill="black")
        buf = io.BytesIO(); img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode()
        r = client.chat.completions.create(
            model=NVIDIA_MODEL,
            messages=[{"role": "user", "content": [
                {"type": "text", "text": "What text appears in the image? Reply only with the text."},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
            ]}],
            max_tokens=500,
        )
        print("✓ Vision OK. Reply:", (r.choices[0].message.content or "").strip()[:80])
    except Exception as e:
        print("❌ The API test failed:", e)
        print("\n- 401/403: wrong key, or it has spaces")
        print("- 404: the model does not exist (the list is below if it could be fetched)")
        print("- 429: free limit reached, wait a bit")
        if "404" in str(e):
            try:
                ids = sorted(m.id for m in client.models.list().data)
                print("\nAvailable models:\n" + "\n".join(ids))
            except Exception as e2:
                print("Could not list the models:", e2)

## 5. Upload one or several documents

Select **one file** to convert just that one, or **several** (Ctrl or Shift in the dialog) to convert them all in a batch. Running this cell again discards the previous uploads.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

INPUT_DIR = Path("/content/input")
shutil.rmtree(INPUT_DIR, ignore_errors=True)
INPUT_DIR.mkdir(parents=True)

uploaded = files.upload()
assert uploaded, "No file was uploaded"

# Marker saves each document in a folder named after the file without its extension,
# so "topic.pdf" and "topic.docx" need different names.
input_files, used_stems = [], set()
for name in uploaded:
    src = Path(name)
    stem, n = src.stem, 2
    while stem.lower() in used_stems:
        stem, n = f"{src.stem}_{n}", n + 1
    used_stems.add(stem.lower())
    dest = INPUT_DIR / f"{stem}{src.suffix}"
    shutil.move(str(src), str(dest))
    input_files.append(dest)

mode = "one document" if len(input_files) == 1 else f"batch of {len(input_files)} documents"
print(f"\n✓ Mode: {mode}")
for f in input_files:
    print("  -", f.name)

## 6. Convert

With one document it uses `marker_single`; with several, `marker`, which loads the models only once for all of them. At the end it lists the documents that failed, if any.

In [ ]:
import shutil, subprocess
from importlib.metadata import version
from pathlib import Path

assert globals().get("input_files"), "Upload the documents first (step 5)"
assert version("marker-pdf").startswith("1."), "marker-pdf 2.x needs Docker. Run step 2 again."
NVIDIA_API_KEY = globals().get("NVIDIA_API_KEY", "")

OUTPUT_DIR = Path("/content/output")
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
OUTPUT_DIR.mkdir(parents=True)

if len(input_files) == 1:
    cmd = ["marker_single", str(input_files[0])]
else:
    # With more than one worker the T4 runs out of memory
    cmd = ["marker", str(INPUT_DIR), "--workers", "1"]

cmd += ["--output_format", OUTPUT_FORMAT, "--output_dir", str(OUTPUT_DIR)]
if FORCE_OCR:
    cmd.append("--force_ocr")
if PAGE_RANGE:
    cmd += ["--page_range", PAGE_RANGE]
if USE_LLM:
    assert NVIDIA_API_KEY, "Missing API key: run step 4"
    cmd += ["--use_llm",
            "--llm_service", "marker.services.openai.OpenAIService",
            "--openai_base_url", NVIDIA_BASE_URL,
            "--openai_model", NVIDIA_MODEL,
            "--openai_api_key", NVIDIA_API_KEY]
    if REDO_INLINE_MATH:
        cmd.append("--redo_inline_math")

print("Running:", " ".join("***" if NVIDIA_API_KEY and a == NVIDIA_API_KEY else a for a in cmd), "\n")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()

# In batch mode, Marker carries on if a document fails and still exits with code 0
out_ext = {"markdown": "md", "html": "html", "json": "json"}[OUTPUT_FORMAT]
converted = [f for f in input_files if (OUTPUT_DIR / f.stem / f"{f.stem}.{out_ext}").exists()]
failed = [f for f in input_files if f not in converted]

print()
if converted:
    print(f"✓ Converted: {len(converted)} of {len(input_files)}")
if failed:
    print(f"❌ Failed ({len(failed)}): " + ", ".join(f.name for f in failed))
    print("   Search for its name in the log above to see the error.")
if proc.returncode != 0:
    print(f"❌ Marker exited with an error (code {proc.returncode}). Check the log above.")

## 7. Package and download

Builds a ZIP with one folder per document: `name/name.md` and `name/images/`. Image links are rewritten to `images/...` and equations end up as `$...$` (inline) and `$$...$$` (block). Unzip it inside your Obsidian vault.

With `html` or `json`, the ZIP contains Marker's output unchanged. You can run this cell again without converting again.

In [ ]:
import re, shutil, urllib.parse
from pathlib import Path
from google.colab import files

assert globals().get("converted"), "No converted documents. Run the conversion first (step 6)."

IMG_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".gif"}
MD_IMAGE = re.compile(r"!\[([^\]]*)\]\(([^)]+)\)")


def obsidian_name(stem):
    return re.sub(r'[\\/:*?"<>|#^\[\]]', "_", stem).strip() or "document"


def to_obsidian(md_path, dest):
    # Returns (text, number of images, broken links)
    src_dir = md_path.parent
    imgs = {f.name: f for f in src_dir.iterdir() if f.suffix.lower() in IMG_EXTS}
    dest.mkdir(parents=True)
    if imgs:
        (dest / "images").mkdir()
    for name, f in imgs.items():
        shutil.copy(f, dest / "images" / name)

    broken = []
    def fix_img(m):
        alt, target = m.group(1), urllib.parse.unquote(m.group(2).strip())
        name = Path(target).name
        if name in imgs:
            return f"![{alt}](images/{urllib.parse.quote(name)})"
        broken.append(target)
        return m.group(0)

    text = md_path.read_text(encoding="utf-8")
    text = MD_IMAGE.sub(fix_img, text)
    # Equations: Obsidian only understands $...$ and $$...$$
    text = re.sub(r"\\\[(.+?)\\\]", lambda m: "\n$$" + m.group(1).strip() + "$$\n", text, flags=re.S)
    text = re.sub(r"\\\((.+?)\\\)", lambda m: "$" + m.group(1).strip() + "$", text, flags=re.S)
    text = re.sub(r'<span id="[^"]*"></span>', "", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip() + "\n"

    (dest / f"{dest.name}.md").write_text(text, encoding="utf-8")
    return text, len(imgs), broken


BASE = Path("/content/package")
shutil.rmtree(BASE, ignore_errors=True)
BASE.mkdir(parents=True)

used_names = set()
for f in converted:
    name, n = obsidian_name(f.stem), 2
    base_name = name
    while name.lower() in used_names:
        name, n = f"{base_name}_{n}", n + 1
    used_names.add(name.lower())
    dest = BASE / name

    if OUTPUT_FORMAT != "markdown":
        shutil.copytree(OUTPUT_DIR / f.stem, dest)
        print(f"✓ {name}/ ({OUTPUT_FORMAT})")
        continue

    text, n_imgs, broken = to_obsidian(OUTPUT_DIR / f.stem / f"{f.stem}.md", dest)
    n_block = len(re.findall(r"\$\$", text)) // 2
    n_inline = len(re.findall(r"(?<!\$)\$(?!\$)[^$\n]+\$(?!\$)", text))
    print(f"✓ {name}/{name}.md  ({len(text):,} characters, {n_imgs} images, "
          f"{n_block} block and ~{n_inline} inline equations)")
    if broken:
        print(f"  ⚠️ {len(broken)} link(s) to images that do not exist: {broken[:5]}")
    if not n_imgs:
        print("  ⚠️ Marker extracted no images (PAGE_RANGE with few pages, or no detectable figures?)")

suffix = "obsidian" if OUTPUT_FORMAT == "markdown" else OUTPUT_FORMAT
zip_name = f"{name}_{suffix}" if len(converted) == 1 else f"documents_{suffix}"
zip_path = shutil.make_archive(f"/content/{zip_name}", "zip", root_dir=BASE)
print(f"\n📦 {Path(zip_path).name}: {len(converted)} document(s)")
files.download(zip_path)